## Detecting inversions with local PCA

This notebook loads the results of windowed local PCA for 540 _An. baimaii_ samples from Southeast Asia. By fiddling with the contig order and orientation, we want to make the plots look nice.

Let's start with some setup.

In [ ]:
# Set up environment
import numpy as np
import pandas as pd
import plotly.express as px
from tqdm.notebook import tqdm
#import sgkit as sg
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Set plot style
sns.set(style="whitegrid")

# Load per-sample metadata
metadata = pd.read_csv('/Users/dennistpw/github/vobs-asia/work/dirus-popgen-2025-05-20/baimaii-metadata.csv')

# This dictionary maps the country of collection to the colours that we want to use to plot them with.
colour_pairs = {
    'Bangladesh': ['#e31a1c', '#fb9a99'],
    'Thailand': ['#33a02c', '#b2df8a'],
    'Cambodia': ['#1f78b4', '#a6cee3']
}

pca_dir = "/Users/dennistpw/github/vobs-asia/work/dirus-popgen-2025-05-20/local_pca/" # This sets the directory the data are in (to this directory)

ymin, ymax = -200, 1000 # This is the max axis values on the plots.

# This is the function for loading and plotting the data.

def run_plots():
    # --- LOAD & CONCATENATE ALL CONTIG PCA DFS ---
    all_dfs = []
    for fname in os.listdir(pca_dir):
        if fname.endswith("-localpca.csv"):
            contig = fname.replace("-localpca.csv", "")
            df = pd.read_csv(os.path.join(pca_dir, fname))
            df["contig"] = contig
            all_dfs.append(df)

    if not all_dfs:
        raise FileNotFoundError("No local_pca/*.csv files found.")

    df_plot = pd.concat(all_dfs, ignore_index=True)
    print(f"Loaded {len(df_plot):,} rows from {len(all_dfs)} contigs.")

    # merge metadata if needed
    if "country" not in df_plot.columns:
        df_plot = df_plot.merge(
            metadata.reset_index().rename(columns={"index": "sample_index"}),
            on="sample_index",
            how="left"
        )

    # ensure numeric positions
    if "pos" not in df_plot.columns:
        df_plot = df_plot.rename(columns={"midpoint": "pos"})

    # --- SORT AND CREATE CUMULATIVE GENOME POSITION ---
    contig_lengths = df_plot.groupby("contig")["pos"].max()
    sorted_lengths = contig_lengths.reindex(user_contig_order).fillna(0)
    contig_offsets = sorted_lengths.cumsum().shift(fill_value=0)

    df_plot["contig"] = pd.Categorical(df_plot["contig"], categories=user_contig_order, ordered=True)
    df_plot = df_plot.sort_values(["contig", "pos"]).copy()
    df_plot["contig_offset"] = df_plot["contig"].map(contig_offsets).astype(float)
    df_plot["genome_position"] = df_plot["pos"] + df_plot["contig_offset"]

    # --- FILTER FOR PC1 AND REMOVE BAD SAMPLE ---
    df_plot = df_plot.query('PC == "PC1"')
    df_plot = df_plot.query('sample_id != "VBS45974-6296STDY9478582"')

    # --- APPLY PER-CONTIG FLIPS ---
    for contig, opts in contig_options.items():
        mask = df_plot["contig"] == contig
        if mask.any():
            if opts.get("reverse_x", False):
                max_pos = df_plot.loc[mask, "genome_position"].max()
                min_pos = df_plot.loc[mask, "genome_position"].min()
                df_plot.loc[mask, "genome_position"] = max_pos - (df_plot.loc[mask, "genome_position"] - min_pos)
            if opts.get("reverse_y", False):
                # Center around the mean of this contig
                mean_val = df_plot.loc[mask, "PC_value"].mean()
                df_plot.loc[mask, "PC_value"] = 2*mean_val - df_plot.loc[mask, "PC_value"]

    # --- PREPARE CONTIG MIDPOINTS ---
    contig_midpoints = (sorted_lengths / 2 + contig_offsets).to_dict()

    # --- PLOT PC1 PER COUNTRY ---
    plot_dir = pca_dir + "/plots"
    os.makedirs(plot_dir, exist_ok=True)
    countries = ["Bangladesh", "Thailand", "Cambodia"]

    for country in countries:
        df_country = df_plot.query('country == @country').copy()

        # only include contigs present in this country
        present_contigs = [c for c in user_contig_order if c in df_country["contig"].unique()]
        contig_to_colour = {contig: colour_pairs[country][i % 2] for i, contig in enumerate(present_contigs)}
        df_country["plot_colour"] = df_country["contig"].map(contig_to_colour)

        import plotly.express as px

        fig = px.line(
            df_country,
            x="genome_position",
            y="PC_value",
            line_group="sample_id",
            color="contig",
            color_discrete_map=contig_to_colour,
            title=f"{country}",
        )

        # contig background rectangles
        bg_colors = ["#ffffff", "#d9d9d9"]
        for i, contig in enumerate(present_contigs):
            start = contig_offsets[contig]
            end = start + sorted_lengths[contig]
            fig.add_shape(
                type="rect",
                x0=start,
                x1=end,
                y0=ymin,
                y1=ymax,
                fillcolor=bg_colors[i % 2],
                line=dict(width=0),
                layer="below"
            )

        # update axes
        fig.update_layout(
            template="simple_white",
            width=1000,  
            height=300,
            showlegend=False,
            xaxis=dict(
                title=None,
                tickmode="array",
                tickvals=[(contig_offsets[c]+sorted_lengths[c]/2) for c in present_contigs],
                ticktext=present_contigs,
                tickangle=90,
                showline=False,
                showgrid=False,
                ticks="",
                showticklabels=False,
                tickfont=dict(size=14)
            ),
            yaxis=dict(title="PC1", range=[ymin, ymax])
        )
        fig.write_image(os.path.join(plot_dir, f"local_pca_PC1_{country}.png"), width=1200, height=300)
        fig.show()


In [ ]:
def run_plots_mpl():
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    # --- LOAD & CONCATENATE ALL CONTIG PCA DFS ---
    all_dfs = []
    for fname in os.listdir(pca_dir):
        if fname.endswith("-localpca.csv"):
            contig = fname.replace("-localpca.csv", "")
            df = pd.read_csv(os.path.join(pca_dir, fname))
            df["contig"] = contig
            all_dfs.append(df)

    if not all_dfs:
        raise FileNotFoundError("No local_pca/*.csv files found.")

    df_plot = pd.concat(all_dfs, ignore_index=True)
    print(f"[MPL] Loaded {len(df_plot):,} rows from {len(all_dfs)} contigs.")

    # merge metadata
    if "country" not in df_plot.columns:
        df_plot = df_plot.merge(
            metadata.reset_index().rename(columns={"index": "sample_index"}),
            on="sample_index",
            how="left"
        )

    # ensure numeric 'pos'
    if "pos" not in df_plot.columns:
        df_plot = df_plot.rename(columns={"midpoint": "pos"})

    # --- SORT AND CREATE CUMULATIVE GENOME POSITION ---
    contig_lengths = df_plot.groupby("contig")["pos"].max()
    sorted_lengths = contig_lengths.reindex(user_contig_order).fillna(0)
    contig_offsets = sorted_lengths.cumsum().shift(fill_value=0)

    df_plot["contig"] = pd.Categorical(df_plot["contig"], categories=user_contig_order, ordered=True)
    df_plot = df_plot.sort_values(["contig", "pos"]).copy()
    df_plot["contig_offset"] = df_plot["contig"].map(contig_offsets).astype(float)
    df_plot["genome_position"] = df_plot["pos"] + df_plot["contig_offset"]

    # --- FILTER PC1 & REMOVE OUTLIER ---
    df_plot = df_plot.query('PC == "PC1"')
    df_plot = df_plot.query('sample_id != "VBS45974-6296STDY9478582"')

    # --- APPLY CONTIG-BASED FLIPS ---
    for contig, opts in contig_options.items():
        mask = df_plot["contig"] == contig
        if mask.any():
            if opts.get("reverse_x", False):
                max_pos = df_plot.loc[mask, "genome_position"].max()
                min_pos = df_plot.loc[mask, "genome_position"].min()
                df_plot.loc[mask, "genome_position"] = max_pos - (df_plot.loc[mask, "genome_position"] - min_pos)
            if opts.get("reverse_y", False):
                mean_val = df_plot.loc[mask, "PC_value"].mean()
                df_plot.loc[mask, "PC_value"] = 2*mean_val - df_plot.loc[mask, "PC_value"]

    # --- PLOTTING ---
    plot_dir = os.path.join(pca_dir, "plots_mpl")
    os.makedirs(plot_dir, exist_ok=True)

    countries = ["Bangladesh", "Thailand", "Cambodia"]

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(12, 8),
        dpi=150,
        sharex=True
    )

    bg_colors = ["#ffffff", "#d9d9d9"]

    for ax, country in zip(axes, countries):

        df_country = df_plot.query('country == @country').copy()

        # contigs present for this country
        present_contigs = [
            c for c in user_contig_order
            if c in df_country["contig"].unique()
        ]

        # two alternating colors per country
        contig_to_colour = {
            contig: colour_pairs[country][i % 2]
            for i, contig in enumerate(present_contigs)
        }

        df_country["plot_colour"] = df_country["contig"].map(contig_to_colour)

        # --- background rectangles ---
        for i, contig in enumerate(present_contigs):
            start = contig_offsets[contig]
            end = start + sorted_lengths[contig]
            ax.add_patch(Rectangle(
                (start, ymin),
                end - start,
                ymax - ymin,
                facecolor=bg_colors[i % 2],
                edgecolor='none',
                zorder=0
            ))

        # --- per-sample lines ---
        for sid, dsub in df_country.groupby("sample_id"):
            ax.plot(
                dsub["genome_position"],
                dsub["PC_value"],
                lw=0.5,
                color=dsub["plot_colour"].iloc[0],
                alpha=0.8
            )

        ax.set_ylim(ymin, ymax)
        ax.set_ylabel("PC1")
        ax.set_title(country)

        # remove x-ticks
        ax.set_xticks([])
        ax.set_xticklabels([])

    # only bottom panel gets an x-label
    axes[-1].set_xlabel("Genomic position (cumulative)")

    plt.tight_layout()

    out_svg = os.path.join(plot_dir, "local_pca_PC1_allcountries.svg")
    fig.savefig(out_svg, format="svg")
    print(f"[MPL] Wrote SVG → {out_svg}")


### The fiddly bit that Martin and I don't want to do.

The genome data are organised into contigs. A contig is basically a piece of genome sequence that hasn't been scaffolded into a chromosome. Because the _An_ baimaii reference genome is quite fragmented, we are dealing with lots of bitty contigs, as opposed to a nice chromosomal referne genome that is organised into, say, chromosomes 2,3 and X. We don't know what order the contigs need to be in, so we are doing the cheapo way of trying this manually. The list of contigs is below - they will be plotted in this order. 

In [ ]:
# CONTIG ORDER
user_contig_order = [
    'KB672491','KB672824','KB672490', 'KB672902', 'KB672791', 'KB672602', 'KB672835',
    'KB672868', 'KB672713', 'KB673645','KB672980', 'KB672802','KB672924','KB673090',
    'KB673201', 'KB673312', 'KB672979', 'KB673423', 'KB672957', 'KB673534','KB672846','KB672991','KB672857',
    'KB672813',
    'KB672869', 'KB672880', 'KB672891', 'KB672913', 
    'KB672935', 'KB672946', 'KB672968', 
    'KB673002', 
    'KB673035', 'KB673057', 'KB673024', 'KB673046', 'KB673068', 'KB673079', 'KB673091',
    'KB673102', 'KB673124', 'KB673113', 'KB673157', 'KB673146',
    'KB673135', 'KB673168', 'KB673190', 'KB673179', 'KB673202',
    'KB673213', 'KB673235', 'KB673224', 'KB673268', 'KB673246',
    'KB673257', 'KB673279', 'KB673301', 'KB673290', 'KB673313',
    'KB673324', 'KB673357', 'KB673379', 'KB673346', 'KB673457',
    'KB673335', 'KB673535', 'KB673546', 'KB673435', 'KB673446',
    'KB673368', 'KB673424', 'KB673390', 'KB673401', 'KB673468',
    'KB673479','KB673646', 'KB673412',  'KB673490', 'KB673013',
]


# CONTIG FLIPS (mark contigs to reverse)
contig_options = {c: {"reverse_x": False, "reverse_y": False} for c in user_contig_order}
contig_options["KB672490"]["reverse_y"] = True
contig_options["KB673090"]["reverse_y"] = True
contig_options["KB673534"]["reverse_x"] = True
contig_options["KB672491"]["reverse_x"] = True
contig_options["KB672802"]["reverse_y"] = True
contig_options["KB672991"]["reverse_y"] = True
contig_options["KB672991"]["reverse_x"] = True
contig_options["KB672857"]["reverse_y"] = True
contig_options["KB672857"]["reverse_x"] = True
contig_options["KB672868"]["reverse_x"] = True
contig_options["KB672957"]["reverse_x"] = True
contig_options["KB672902"]["reverse_y"] = True
contig_options["KB673201"]["reverse_y"] = True
contig_options["KB672980"]["reverse_y"] = True
contig_options["KB672924"]["reverse_y"] = True
contig_options["KB672924"]["reverse_x"] = True
contig_options["KB673046"]["reverse_x"] = True
contig_options["KB673024"]["reverse_x"] = True
contig_options["KB673312"]["reverse_y"] = True
contig_options["KB672813"]["reverse_y"] = True




### The plots
Run the code below when you want to generate the plots

In [ ]:
run_plots()